# LAB- P3: La Decisión Óptima de Consumo-Ahorro (Julia)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P3-v1.0
*   **Capítulo de Referencia**: Capítulo 4, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Analizar la decisión óptima intertemporal de un hogar en un ciclo de vida finito. Comprender cómo la tasa de interés real, la impaciencia subjetiva y la estructura temporal de ingresos determinan el perfil óptimo de consumo y la trayectoria de acumulación de activos financieros (ahorro/deuda).

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** el carácter intertemporal de la elección del consumidor y cómo el ahorro permite disociar el perfil de consumo del de ingresos.
2.  **Modelar** el problema de optimización dinámica del consumidor en tiempo discreto sujeto a restricciones presupuestarias secuenciales.
3.  **Analizar e Interpretar** la Ecuación de Euler y la condición de estado estacionario ($\bar{R} = \theta$).
4.  **Resolver** sistemas de ecuaciones no lineales (FOC) usando `fsolve` y problemas de optimización convexa directamente en Python usando `solve_direct_optim`.
5.  **Evaluar** los efectos de shocks sobre los flujos de ingresos (salarios crecientes, jubilación) y de cambios en los parámetros del modelo ($\beta, R$) sobre los patrones de ahorro y deuda.


## 1. El Marco Teórico: Optimización Intertemporal del Consumidor

El consumidor decide su nivel de consumo a lo largo de un ciclo de vida finito de $T$ periodos ($t = 0, 1, \dots, T-1$). Su objetivo es maximizar la suma descontada de utilidades subjetivas:

$$\max_{\{C_t\}_{t=0}^{T-1}} \sum_{t=0}^{T-1} \beta^t U(C_t)$$

Donde:
*   $C_t$ es el consumo en el periodo $t$.
*   $U(C_t)$ es la utilidad instantánea. Asumimos una **utilidad logarítmica** $U(C_t) = \ln(C_t)$, que cumple con ser estrictamente cóncava ($U_C > 0, U_{CC} < 0$, indicando utilidad marginal decreciente).
*   $\beta = \frac{1}{1+\theta}$ es el **factor de descuento intertemporal**, con $\theta > 0$ representando la tasa de preferencia temporal subjetiva (impaciencia).

### 1.1 Restricciones Presupuestarias y Condiciones de Contorno
La maximización está sujeta a la restricción presupuestaria en cada periodo:

$$C_t + B_t = (1 + R_{t-1})B_{t-1} + W_t$$

Donde:
*   $B_t$ es el stock de activos financieros al final del periodo $t$ (si es positivo, es acreedor/ahorro; si es negativo, es deudor/deuda).
*   $R_{t-1}$ es la tasa de interés real de los activos en el periodo anterior.
*   $W_t$ es el ingreso salarial exógeno en el periodo $t$.

Además, asumimos las siguientes condiciones de contorno:
1.  **Sin activos iniciales**: $B_{-1} = 0$, lo que implica que en el periodo $t=0$, $B_0 = W_0 - C_0$.
2.  **Condición Terminal**: $B_{T-1} = 0$, es decir, el consumidor no deja deudas ni herencias al final de su vida ($t = T-1$).

### 1.2 La Ecuación de Euler
Resolviendo el problema mediante el método de Lagrange, se obtiene la **Ecuación de Euler**, que gobierna la trayectoria óptima del consumo:

$$U_C(C_t) = \beta (1 + R_t) U_C(C_{t+1})$$

Para la utilidad logarítmica, la utilidad marginal es $U_C(C) = 1/C$, lo que reduce la ecuación de Euler a:

$$C_{t+1} = \beta (1 + R_t) C_{t}$$

Esta relación revela que:
*   Si $\beta (1+R) > 1$ (el interés real supera la impaciencia $\theta$), el consumo tiene **pendiente positiva** (crece con el tiempo).
*   Si $\beta (1+R) < 1$ (la impaciencia supera el interés real), el consumo tiene **pendiente negativa** (decrece).
*   Si $\beta (1+R) = 1$ (estado estacionario, $R = \theta$), el consumo es constante en todos los periodos.


In [ ]:
# Este cuaderno depende del paquete `MacroAIComp` (Project.toml/Manifest.toml
# en la raíz del repositorio). En MyBinder (ver docs/DESPLIEGUE_BINDER.md) y en
# tu entorno local, el kernel ya arranca dentro del repositorio clonado, así
# que la celda siguiente activa e instancia el proyecto automáticamente.
# Nota: Google Colab no soporta Julia de forma nativa desde un notebook .ipynb;
# para la versión Julia de esta práctica usa MyBinder.


In [ ]:
using Pkg
Pkg.activate("../..")
Pkg.instantiate()

using MacroAIComp
using Plots
import Plots: mm
default(gridalpha=0.6, gridstyle=:dot)  # estilo de grid consistente con la versión Python
using LinearAlgebra
using Interact
using BenchmarkTools


## 2. Métodos de Resolución Computacional: FOC vs Optimización Directa

En macroeconomía aplicada, podemos resolver este tipo de modelos de dos formas:
1.  **Resolución de FOC (`fsolve`)**: Planteando las $T$ ecuaciones no lineales de Euler y la condición terminal $B_{T-1}=0$ para hallar el vector de consumo $C$.
2.  **Optimización Convexa Directa (`solve_direct_optim`)**: Maximizar directamente la función objetivo bajo restricciones analíticas lineales.

A continuación, ejecutaremos ambos métodos con la calibración base y un salario constante $W=10$ para comprobar su equivalencia numérica.

*Nota sobre la errata de MATLAB:* La formulación en `fsolve` incluye la corrección terminal del salario ($W_T$) para forzar a que el saldo final $B_T$ sea exactamente $0.0$, emulando el Solver de Excel.


In [ ]:
params = default_calibration(ConsumptionSavingParameters)

println("CALIBRACIÓN ECONÓMICA BASE:")
println("-"^50)
println("  Duración del ciclo de vida (T)  : ", params.T, " periodos")
println("  Factor de descuento (beta)      : ", params.beta, " (equivale a theta ≈ ", round((1-params.beta)/params.beta*100, digits=2), "%)")
println("  Tasa de interés real (R)        : ", round(params.R*100, digits=2), "%")
println("-"^50)


## 2.1 Verificación frente al oráculo

Comparamos contra los valores reportados en el libro y reproducidos por el
código MATLAB del Apéndice G (`referencia/consumption.m`), recogidos en
`oraculo.md`:

| Magnitud | Valor esperado (oráculo) |
|---|---|
| Activos terminales $B_{T-1}$ (cualquier perfil) | 0.0 (tol. 1e-6) |
| Equivalencia fsolve vs optimización directa | $C$ y $B$ idénticos (rtol 1e-4) |
| Perfil creciente — $B_0$ | Negativo (endeudamiento juvenil) |
| Perfil creciente — pendiente de $C$ | Negativa ($\beta(1+R)<1$) |
| Perfil jubilación — pico de activos | $t=19$ (fin de vida laboral) |
| Perfil jubilación — activos en vida laboral | Positivos (acumulación) |
| $\beta=0.99$ — pendiente de $C$ | Positiva ($\beta(1+R)>1$) |

Así puedes comparar a simple vista, sin abrir `oraculo.md`, el número que
debería salir en cada celda siguiente con el que realmente sale.

In [ ]:
# Generar salario constante
W_const = generate_income_profile("constant", params.T)
println("W constante: ", W_const[1:5], "...")

res_foc = solve_foc_fsolve(params, W_const)
res_opt = solve_direct_optim(params, W_const)

println("-"^75)
println("  Consumo Inicial C(0) [fsolve]   : ", round(res_foc["C"][1], digits=6))
println("  Consumo Inicial C(0) [optim]    : ", round(res_opt["C"][1], digits=6))
println("  Consumo Final C(T-1) [fsolve]   : ", round(res_foc["C"][end], digits=6))
println("  Consumo Final C(T-1) [optim]    : ", round(res_opt["C"][end], digits=6))
println("  Activos Finales B(T-1) [fsolve] : ", res_foc["B"][end])
println("  Activos Finales B(T-1) [optim]  : ", res_opt["B"][end])
println("-"^75)

diferencia_max = maximum(abs.(res_foc["C"] .- res_opt["C"]))
println("Máxima diferencia absoluta en el consumo: ", diferencia_max)
if diferencia_max < 1e-5
    println("✅ ¡Los solucionadores son equivalentes numéricamente!")
else
    println("❌ Hay diferencias entre solucionadores.")
end


## 2.1 Verificación frente al oráculo

Comparamos contra los valores reportados en el libro y reproducidos por el
código MATLAB del Apéndice G (`referencia/consumption.m`), recogidos en
`oraculo.md`:

| Magnitud | Valor esperado (oráculo) |
|---|---|
| Activos terminales $B_{T-1}$ (cualquier perfil) | 0.0 (tol. 1e-6) |
| Equivalencia fsolve vs optimización directa | $C$ y $B$ idénticos (rtol 1e-4) |
| Perfil creciente — $B_0$ | Negativo (endeudamiento juvenil) |
| Perfil creciente — pendiente de $C$ | Negativa ($\beta(1+R)<1$) |
| Perfil jubilación — pico de activos | $t=19$ (fin de vida laboral) |
| Perfil jubilación — activos en vida laboral | Positivos (acumulación) |
| $\beta=0.99$ — pendiente de $C$ | Positiva ($\beta(1+R)>1$) |

Así puedes comparar a simple vista, sin abrir `oraculo.md`, el número que
debería salir en cada celda siguiente con el que realmente sale.

In [ ]:
# Verificamos la condición terminal (B_{T-1}=0) y la equivalencia entre
# solvers contra el oráculo del Apéndice G (MATLAB) recogido en oraculo.md.
# @assert isapprox compara dos valores y SOLO lanza un error si la
# diferencia supera la tolerancia. No usamos "==" porque la aritmética con
# decimales casi nunca da resultados exactamente iguales (errores de redondeo
# internos). Si el port a Julia tuviera un error, esta celda lanzaría
# AssertionError y detendría la ejecución antes de seguir construyendo
# gráficos sobre un resultado incorrecto.

# Condición terminal: el consumidor no deja deudas ni herencias
@assert isapprox(res_foc["B"][end], 0.0; atol=1e-6)
@assert isapprox(res_opt["B"][end], 0.0; atol=1e-6)

# Equivalencia fsolve vs optimización directa: C y B deben ser idénticos
# (Julia usa isapprox con rtol para equivalencia relativa)
@assert isapprox(res_foc["C"], res_opt["C"]; rtol=1e-4, atol=1e-4)
@assert isapprox(res_foc["B"], res_opt["B"]; rtol=1e-4, atol=1e-4)
println("OK: coincide con el oráculo MATLAB (Apéndice G).")


## 3. Detrás de la Escena: El Sistema de Ecuaciones FOC de la Ecuación de Euler

Para comprender cómo funciona el resolvedor `fsolve`, es fundamental observar las ecuaciones que plantea internamente la función. A continuación se detalla la lógica de la función residual:

*   **Euler ($t = 0 \dots T-2$):** Para cada periodo, se debe cumplir la ecuación de Euler:
    $$f(t) = C_{t+1} - \beta (1 + R) C_t = 0$$
*   **Activos ($B_t$):** Se calcula de forma secuencial la acumulación:
    $$B_0 = W_0 - C_0$$
    $$B_t = (1+R)B_{t-1} + W_t - C_t$$
*   **Condición Terminal ($t = T-1$):** Para asegurar que $B_{T-1} = 0$, la última ecuación del sistema obliga a que el consumo final agote la riqueza restante:
    $$f(T-1) = C_{T-1} - (1+R)B_{T-2} - W_{T-1} = 0$$

Esto da un sistema cuadrado de $T$ ecuaciones con $T$ incógnitas ($C_0, \dots, C_{T-1}$), que `fsolve` resuelve iterativamente mediante una aproximación lineal del Jacobiano (método de tipo Newton-Raphson).


## 4. Simulación Interactiva y Respuesta ante Shocks

A continuación, implementaremos la visualización didáctica en **3 paneles**:
*   **Panel 1 (Consumo e Ingresos)**: Muestra la trayectoria de consumo óptimo $C_t$ contra el ingreso salarial $W_t$.
*   **Panel 2 (Activos Financieros)**: Muestra la acumulación de riqueza o endeudamiento $B_t$. El área sombreada en azul representa una posición acreedora (ahorro acumulado) y en naranja una posición deudora (deuda).
*   **Panel 3 (Utilidad Descontada)**: Muestra la contribución de cada periodo al bienestar total ($\beta^t \ln C_t$).

Podrás interactuar con los deslizadores para modificar:
1.  **Dinero/Paciencia ($\beta$)**: Factor de descuento.
2.  **Tasa de Interés ($R$)**: Rendimiento del capital.
3.  **Perfil de Ingresos (W)**:
    *   `constant`: Ingreso constante de 10.
    *   `increasing`: Salario creciente (representa acumulación de experiencia).
    *   `retirement`: Jubilación a partir del periodo 20 (salario cae a 0).


In [ ]:
# Simulación interactiva con Interact.jl
@manipulate for beta_val in slider([0.90:0.01:0.99; 0.999]; value=0.97, label="Paciencia (β)"), R_val in slider(-0.05:0.01:0.15; value=0.02, label="Interés (R)"), profile in Widgets.dropdown(["constant", "increasing", "retirement"]; value="constant", label="Perfil Salarial")
    
    params_interactive = ConsumptionSavingParameters(30, beta_val, R_val, 0.0)
    W = generate_income_profile(profile, params_interactive.T)
    res = solve_foc_fsolve(params_interactive, W)
    
    t_axis = 0:(params_interactive.T - 1)
    
    # Panel 1: Consumo e Ingresos
    p1 = plot(t_axis, res["C"], color="#7A3E9F", lw=2.5, label="Consumo (C)")
    plot!(t_axis, res["W"], color="#8EAD3A", lw=2.5, ls=:dash, label="Ingreso (W)")
    title!("Consumo e Ingresos a lo largo del Ciclo de Vida")
    xlabel!("Periodo (t)")
    ylabel!("Unidades de Bienes")
    
    # Panel 2: Activos
    p2 = plot(t_axis, res["B"], color="#004C97", lw=2.5, label="Activos (B)")
    hline!([0.0], color=:black, ls=:dot, label="")
    # Fill between (trick in Plots: fillrange=0)
    plot!(t_axis, max.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color="#004C97", lw=0, label="Ahorro")
    plot!(t_axis, min.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color="#D95319", lw=0, label="Deuda")
    title!("Evolución de Activos Financieros")
    xlabel!("Periodo (t)")
    ylabel!("Riqueza Neta")
    
    # Panel 3: Utilidad
    p3 = plot(t_axis, res["U"], color="#D95319", lw=2.0, label="Utilidad")
    title!("Utilidad Descontada por Periodo")
    xlabel!("Periodo (t)")
    ylabel!("Utilidad")
    
    plot(p1, p2, p3, layout=(1,3), size=(1100, 350), 
         plot_title="Decisión Óptima Intertemporal", top_margin=10mm)
end


## 4.1 Verificación de perfiles de ingreso y sensibilidad

Comprobamos contra el oráculo los resultados cualitativos y cuantitativos
para cada perfil de ingreso y el caso de sensibilidad $\beta=0.99$ (Apéndice G,
`oraculo.md`).

In [ ]:
# Verificamos los casos adicionales del oráculo (Apéndice G):

# --- Perfil creciente: endeudamiento juvenil y pendiente negativa del consumo ---
W_inc = generate_income_profile("increasing", params.T)
res_inc = solve_foc_fsolve(params, W_inc)
@assert res_inc["B"][1] < 0 "B[1] debería ser negativo (endeudamiento juvenil) con perfil creciente, pero es $(res_inc["B"][1])"
# Pendiente de C negativa: C[1] > C[end] porque beta*(1+R) = 0.9894 < 1
@assert res_inc["C"][1] > res_inc["C"][end] "La pendiente del consumo debería ser negativa con beta=0.97, R=0.02"
@assert isapprox(res_inc["B"][end], 0.0; atol=1e-6)
println("  Perfil creciente: B[1]=", round(res_inc["B"][1], digits=4),
        " (negativo), C[1]=", round(res_inc["C"][1], digits=4),
        " > C[end]=", round(res_inc["C"][end], digits=4), " OK")

# --- Perfil jubilación: pico de activos en t=19 (índice 20 en Julia) ---
W_ret = generate_income_profile("retirement", params.T)
res_ret = solve_foc_fsolve(params, W_ret)
peak_idx = argmax(res_ret["B"])
@assert peak_idx == 20 "El pico de activos debería estar en t=19 (índice 20 en Julia), no en $(peak_idx)"
# Activos positivos durante la vida laboral (t < 20, índices 1:20 en Julia)
@assert all(res_ret["B"][1:20] .> 0) "Los activos deberían ser positivos durante la vida laboral"
@assert isapprox(res_ret["B"][end], 0.0; atol=1e-6)
println("  Perfil jubilación: pico de activos en t=", peak_idx - 1,
        ", activos positivos en vida laboral OK")

# --- Sensibilidad beta=0.99: pendiente del consumo positiva ---
params_beta99 = ConsumptionSavingParameters(30, 0.99, 0.02, 0.0)
W_c2 = generate_income_profile("constant", params_beta99.T)
res_beta99 = solve_foc_fsolve(params_beta99, W_c2)
@assert res_beta99["C"][1] < res_beta99["C"][end] "La pendiente del consumo debería ser positiva con beta=0.99 (beta*(1+R)=1.0098>1)"
@assert isapprox(res_beta99["B"][end], 0.0; atol=1e-6)
println("  beta=0.99: C[1]=", round(res_beta99["C"][1], digits=4),
        " < C[end]=", round(res_beta99["C"][end], digits=4),
        " (pendiente positiva) OK")

println("OK: todos los perfiles coinciden con el oráculo MATLAB (Apéndice G).")


## 4.1 Verificación de perfiles de ingreso y sensibilidad

Comprobamos contra el oráculo los resultados cualitativos y cuantitativos
para cada perfil de ingreso y el caso de sensibilidad $\beta=0.99$ (Apéndice G,
`oraculo.md`).

## 5. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Higiene del Sistema de Optimización**: El modelo evita declarar variables globales. Los resolvedores reciben un dataclass `ConsumptionSavingParameters` facilitando la escalabilidad del modelo.
2.  **Modularización Externa**: El "motor matemático" se aloja de forma independiente en `src/macroaicomp/models/consumption_savings.jl`, permitiendo verificar la corrección del solver con Test.jl antes de renderizar la interfaz de usuario.
3.  **Control de Versiones Limpio**: Para cumplir con la directiva del proyecto, los metadatos y outputs interactivos de este cuaderno son removidos automáticamente por `nbstripout` antes de confirmar cualquier commit.


## 7. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [ ]:
# Benchmark simulation
@btime solve_foc_fsolve($params, $W_const)
